In [2]:
// notebooks/bronze/01_source_profile.rsnb

:dep serde = { version = "1", features = ["derive"] }
:dep serde_json = "1"
:dep chrono = { version = "0.4", features = ["serde"] }

use std::collections::{BTreeMap, BTreeSet};
use std::fs;

use chrono::{DateTime, NaiveDate, Utc};
use serde::{Deserialize, Serialize};

#[derive(Debug, Clone, Serialize, Deserialize)]
struct BronzeRefRecord {
    ref_id: String,
    ref_name: String,
    ref_type: String,
    is_active: bool,
    updated_at: Option<DateTime<Utc>>,
    run_id: String,
    source_name: String,
    source_file: Option<String>,
    ingested_at: DateTime<Utc>,
    load_date: String,
    record_hash: Option<String>,
}

#[derive(Debug, Clone, Serialize, Deserialize)]
struct BronzeDailyRecord {
    event_id: String,
    ref_id: String,
    event_date: NaiveDate,
    metric_value: f64,
    status: String,
    source_ts: Option<DateTime<Utc>>,
    run_id: String,
    source_name: String,
    source_file: Option<String>,
    ingested_at: DateTime<Utc>,
    load_date: String,
    record_hash: Option<String>,
}

fn read_text(path: &str) -> String {
    fs::read_to_string(path).unwrap_or_else(|e| panic!("failed to read {}: {}", path, e))
}

fn profile_ref(records: &[BronzeRefRecord]) {
    println!("=== Bronze Ref Profile ===");
    println!("row_count        : {}", records.len());

    let active_count = records.iter().filter(|r| r.is_active).count();
    let inactive_count = records.len().saturating_sub(active_count);

    let unique_ref_ids: BTreeSet<_> = records.iter().map(|r| r.ref_id.clone()).collect();
    let unique_ref_types: BTreeSet<_> = records.iter().map(|r| r.ref_type.clone()).collect();

    let blank_ref_ids = records.iter().filter(|r| r.ref_id.trim().is_empty()).count();
    let blank_ref_names = records.iter().filter(|r| r.ref_name.trim().is_empty()).count();
    let blank_ref_types = records.iter().filter(|r| r.ref_type.trim().is_empty()).count();

    println!("unique_ref_ids   : {}", unique_ref_ids.len());
    println!("unique_ref_types : {}", unique_ref_types.len());
    println!("active_count     : {}", active_count);
    println!("inactive_count   : {}", inactive_count);
    println!("blank_ref_ids    : {}", blank_ref_ids);
    println!("blank_ref_names  : {}", blank_ref_names);
    println!("blank_ref_types  : {}", blank_ref_types);

    let mut type_counts: BTreeMap<String, usize> = BTreeMap::new();
    for record in records {
        *type_counts.entry(record.ref_type.clone()).or_insert(0) += 1;
    }

    println!("\nref_type distribution:");
    for (k, v) in type_counts {
        println!("  {} -> {}", k, v);
    }

    println!("\nfirst 3 rows:");
    for record in records.iter().take(3) {
        println!("{:#?}", record);
    }
}

fn profile_daily(records: &[BronzeDailyRecord]) {
    println!("=== Bronze Daily Profile ===");
    println!("row_count         : {}", records.len());

    let unique_event_ids: BTreeSet<_> = records.iter().map(|r| r.event_id.clone()).collect();
    let unique_ref_ids: BTreeSet<_> = records.iter().map(|r| r.ref_id.clone()).collect();
    let unique_statuses: BTreeSet<_> = records.iter().map(|r| r.status.clone()).collect();

    let blank_event_ids = records.iter().filter(|r| r.event_id.trim().is_empty()).count();
    let blank_ref_ids = records.iter().filter(|r| r.ref_id.trim().is_empty()).count();
    let blank_statuses = records.iter().filter(|r| r.status.trim().is_empty()).count();

    let positive_metrics = records.iter().filter(|r| r.metric_value > 0.0).count();
    let zero_metrics = records.iter().filter(|r| r.metric_value == 0.0).count();
    let negative_metrics = records.iter().filter(|r| r.metric_value < 0.0).count();

    let min_metric = records
        .iter()
        .map(|r| r.metric_value)
        .reduce(f64::min)
        .unwrap_or(0.0);

    let max_metric = records
        .iter()
        .map(|r| r.metric_value)
        .reduce(f64::max)
        .unwrap_or(0.0);

    println!("unique_event_ids  : {}", unique_event_ids.len());
    println!("unique_ref_ids    : {}", unique_ref_ids.len());
    println!("unique_statuses   : {}", unique_statuses.len());
    println!("blank_event_ids   : {}", blank_event_ids);
    println!("blank_ref_ids     : {}", blank_ref_ids);
    println!("blank_statuses    : {}", blank_statuses);
    println!("positive_metrics  : {}", positive_metrics);
    println!("zero_metrics      : {}", zero_metrics);
    println!("negative_metrics  : {}", negative_metrics);
    println!("min_metric_value  : {}", min_metric);
    println!("max_metric_value  : {}", max_metric);

    let mut status_counts: BTreeMap<String, usize> = BTreeMap::new();
    for record in records {
        *status_counts.entry(record.status.clone()).or_insert(0) += 1;
    }

    println!("\nstatus distribution:");
    for (k, v) in status_counts {
        println!("  {} -> {}", k, v);
    }

    println!("\nfirst 3 rows:");
    for record in records.iter().take(3) {
        println!("{:#?}", record);
    }
}

In [3]:
let bronze_ref_path = "/workspace/data/bronze/ref_example/load_date=2026-03-15/run_id=56e2508f-dd42-419d-a417-5b4d36ee51c7/bronze_ref.json";
let bronze_daily_path = "/workspace/data/bronze/daily_example/load_date=2026-03-15/run_id=e358cb13-b4b9-4fe5-8685-02013cfc65ae/bronze_daily.json";

println!("bronze_ref_path   = {}", bronze_ref_path);
println!("bronze_daily_path = {}", bronze_daily_path);

bronze_ref_path   = /workspace/data/bronze/ref_example/load_date=2026-03-15/run_id=56e2508f-dd42-419d-a417-5b4d36ee51c7/bronze_ref.json
bronze_daily_path = /workspace/data/bronze/daily_example/load_date=2026-03-15/run_id=e358cb13-b4b9-4fe5-8685-02013cfc65ae/bronze_daily.json


In [4]:
let bronze_ref_content = read_text(bronze_ref_path);
let bronze_daily_content = read_text(bronze_daily_path);

let bronze_ref_records: Vec<BronzeRefRecord> = serde_json::from_str(&bronze_ref_content)?;
let bronze_daily_records: Vec<BronzeDailyRecord> = serde_json::from_str(&bronze_daily_content)?;

profile_ref(&bronze_ref_records);
println!();
profile_daily(&bronze_daily_records);

=== Bronze Ref Profile ===
row_count        : 3
unique_ref_ids   : 3
unique_ref_types : 2
active_count     : 2
inactive_count   : 1
blank_ref_ids    : 0
blank_ref_names  : 0
blank_ref_types  : 0

ref_type distribution:
  region -> 1
  site -> 2

first 3 rows:
BronzeRefRecord {
    ref_id: "R001",
    ref_name: "Alpha",
    ref_type: "site",
    is_active: true,
    updated_at: None,
    run_id: "56e2508f-dd42-419d-a417-5b4d36ee51c7",
    source_name: "ref_example",
    source_file: Some(
        "ref_source.json",
    ),
    ingested_at: 2026-03-15T00:04:42.414905388Z,
    load_date: "2026-03-15",
    record_hash: None,
}
BronzeRefRecord {
    ref_id: "R002",
    ref_name: "Beta",
    ref_type: "site",
    is_active: true,
    updated_at: None,
    run_id: "56e2508f-dd42-419d-a417-5b4d36ee51c7",
    source_name: "ref_example",
    source_file: Some(
        "ref_source.json",
    ),
    ingested_at: 2026-03-15T00:04:42.414905388Z,
    load_date: "2026-03-15",
    record_hash: None,
}
B

In [5]:
assert!(!bronze_ref_records.is_empty());
assert!(!bronze_daily_records.is_empty());

assert!(bronze_ref_records.iter().all(|r| !r.ref_id.trim().is_empty()));
assert!(bronze_daily_records.iter().all(|r| !r.event_id.trim().is_empty()));
assert!(bronze_daily_records.iter().all(|r| !r.ref_id.trim().is_empty()));

println!("basic bronze source profiling checks passed");

basic bronze source profiling checks passed
